In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.optim as optim
import math

# -----------------------------------
# settings
# -----------------------------------
dt = 0.01
eps = 0.01
Time_steps = 1500
iters = 1000

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class LorenzStep(nn.Module):
    def __init__(self, dt=0.01):
        super().__init__()
        self.sigma = 10.0
        self.rho = 28.0
        self.beta = 8.0 / 3.0
        self.dt = dt

    def forward(self, x):
        X = x[:, 0]
        Y = x[:, 1]
        Z = x[:, 2]

        dx = self.sigma * (Y - X)
        dy = X * (self.rho - Z) - Y
        dz = X * Y - self.beta * Z

        # Euler step
        X = X + self.dt * dx
        Y = Y + self.dt * dy
        Z = Z + self.dt * dz

        return torch.stack([X, Y, Z], dim=1)


model = LorenzStep(dt=dt).to(device)

In [ ]:
def loss_fn(A,B):
  #return torch.mean((A - B)**2)
  return torch.norm(A - B, dim=2).mean()

In [ ]:
import pandas as pd

# bounds
low  = torch.tensor([-20.0, -30.0, 0.0], device=device)
high = torch.tensor([20.0, 30.0, 50.0], device=device)
N = 1000   # 200 random points
#T_list = list(range(100, 1501, 100))
T_list = [300,500,900,1300]

avg_sep_list = []
optimized_points_rows = []

for Time_steps in T_list:
    print(f"\nRunning NIO for T = {Time_steps}")

    # -----------------------------------
    # 200 random initial latent points
    # -----------------------------------
    initial_input = torch.randn(N, 3, device=device)

    z_init = initial_input.clone().detach().requires_grad_(True)
    optimizer = optim.Adam([z_init], lr=0.05)

    # -----------------------------------
    # NIO optimization for this T
    # -----------------------------------
    for i in range(iters):
        x0 = low + (high - low) * torch.sigmoid(z_init)

        xA = x0
        xB = x0 + eps

        trajA = []
        trajB = []

        trajA.append(xA)
        trajB.append(xB)

        for t in range(Time_steps):
            xA = model(xA)
            xB = model(xB)

            trajA.append(xA)
            trajB.append(xB)

        trajA = torch.stack(trajA)   # [Time_steps+1, N, 3]
        trajB = torch.stack(trajB)

        # maximize trajectory divergence (change the loss function sign to minimize)
        loss = -loss_fn(trajA, trajB)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if i % 100 == 0:
            print("iter", i, "loss", loss.item())

    # -----------------------------------
    # after NIO, find trajectory separation
    # and save optimized 200 points
    # -----------------------------------
    with torch.no_grad():
        x0 = low + (high - low) * torch.sigmoid(z_init)

        # save optimized 200 starting points for this Time_steps
        x0_cpu = x0.detach().cpu().numpy()

        for point_id in range(N):
            optimized_points_rows.append({
                "Time_steps": Time_steps,
                "point_id": point_id,
                "x0": x0_cpu[point_id, 0],
                "y0": x0_cpu[point_id, 1],
                "z0": x0_cpu[point_id, 2],
            })

        xA = x0
        xB = x0 + eps

        # this part is to get the trajectory after final optimization
        final_trajA = []
        final_trajB = []

        for t in range(Time_steps):
            xA = model(xA)
            xB = model(xB)

            final_trajA.append(xA)
            final_trajB.append(xB)

        final_trajA = torch.stack(final_trajA)   # [Time_steps+1, N, 3]
        final_trajB = torch.stack(final_trajB)   # [Time_steps+1, N, 3]

        # trajectory separation across all time steps
        final_sep = torch.norm(final_trajA - final_trajB, dim=2)   # [Time_steps, N,3]

        # average over all time steps and all 200 optimized points
        avg_sep = final_sep.mean().item()
        avg_sep_list.append(avg_sep)

        print(f"T = {Time_steps}, avg trajectory separation = {avg_sep:.6f}")

# -----------------------------------
# save optimized points to CSV
# -----------------------------------
optimized_points_df = pd.DataFrame(optimized_points_rows)
optimized_points_df.to_csv("nio_optimized_200_max.csv", index=False) # saving the file

print("Saved CSV: nio_optimized_200_points_each_T.csv")

# -----------------------------------
# plot graph
# -----------------------------------
plt.figure(figsize=(8, 5))
plt.plot(T_list, avg_sep_list, marker='o')
plt.xlabel("Time step T")
plt.ylabel("Average trajectory separation")
plt.title("NIO: Average trajectory separation vs T")
plt.grid(True)
plt.show()